# 10 · Humanizer attack

How easy is it to fool the detector? The AI texts of the test set are rewritten with simple word swaps (`humanizer.py`), and every model is run on the original and the rewritten versions. The human texts get the reverse treatment, to see whether formal human writing gets flagged.

Needs a GPU (Kaggle: Settings → Accelerator → GPU) and `HF_TOKEN` as a Kaggle secret (or in `.env`), because the dataset is private. Takes about 20–30 minutes.

In [ ]:
%pip install -q -U transformers datasets huggingface_hub scikit-learn python-dotenv

## Settings

In [ ]:
import os
import shutil
import sys
import urllib.request
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModelForSequenceClassification, AutoTokenizer

HF_DATASET_REPO = "aliissa824/german-academic-ai-text"
MODELS = {   # name -> Hugging Face repo
    "TF-IDF": "aliissa824/german-academic-ai-text-tfidf",
    "DistilBERT German": "aliissa824/german-academic-ai-text-distilbert-german",
    "XLM-RoBERTa base": "aliissa824/german-academic-ai-text-xlm-roberta-base",
    "mmBERT-small": "aliissa824/german-academic-ai-text-mmbert-small",
    "mmBERT-base": "HassanHijazi/german-ai-text-mmbert-base",          # by Hassan
}
MAX_LENGTH = 512
BATCH_SIZE = 32
LABELS = ["human", "ai_generated", "ai_assisted"]
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


def get_secret(name):
    """Kaggle Secrets on Kaggle, the .env file in the repo root anywhere else."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        from dotenv import load_dotenv
        load_dotenv(Path.cwd().parent / ".env")
        return os.getenv(name)


HF_TOKEN = get_secret("HF_TOKEN")

# On Kaggle the notebook runs without the repo: fetch the rewrite rules from GitHub
if not Path("humanizer.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/aliissa824/German-AI-Text-Detector-from-scratch/"
                               "main/10_humanizer-attack/humanizer.py", "humanizer.py")
sys.path.insert(0, str(Path.cwd()))
from humanizer import formalize, humanize

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (this will be slow)")

## Build the versions

- **original:** every test text as it is
- **light / strong:** the AI texts, rewritten by `humanize()`
- **formalized:** the human texts, rewritten by `formalize()`

In [ ]:
test = load_dataset(HF_DATASET_REPO, split="test", token=HF_TOKEN).to_pandas()

rows = []
for r in test.itertuples():
    rows.append({"id": r.id, "label": r.label, "version": "original", "text": r.text, "changes": 0})
    if r.label > 0:
        for level in ["light", "strong"]:
            text, changes = humanize(r.text, level)
            rows.append({"id": r.id, "label": r.label, "version": level, "text": text, "changes": changes})
    else:
        text, changes = formalize(r.text)
        rows.append({"id": r.id, "label": r.label, "version": "formalized", "text": text, "changes": changes})
versions = pd.DataFrame(rows)

summary = versions[versions["version"] != "original"].groupby("version")["changes"].agg(
    texts="size", changed=lambda c: (c > 0).mean(), median_changes="median", mean_changes="mean")
print(summary.to_string(float_format=lambda v: f"{v:.2f}"))

## Run every model on every version

In [ ]:
unique_texts = versions["text"].drop_duplicates().tolist()      # unchanged texts are only predicted once
print(f"{len(versions):,} rows, {len(unique_texts):,} different texts per model")


def to_our_labels(config):
    """Column order of the model's output -> our label ids (Hassan's model uses HUMAN / AI / AI_ASSISTED)."""
    alias = {"human": 0, "ai_generated": 1, "ai": 1, "ai_assisted": 2}
    order = [alias[config.id2label[i].lower()] for i in range(config.num_labels)]
    assert sorted(order) == [0, 1, 2], config.id2label
    return np.array(order)


@torch.inference_mode()
def predict_transformer(repo, texts):
    tokenizer = AutoTokenizer.from_pretrained(repo, token=HF_TOKEN)
    model = AutoModelForSequenceClassification.from_pretrained(repo, token=HF_TOKEN).to(DEVICE).eval()
    if DEVICE == "cuda":
        model = model.half()
    order = to_our_labels(model.config)
    by_length = np.argsort([len(t) for t in texts])             # similar lengths in a batch = less padding
    predictions = np.empty(len(texts), dtype=int)
    for start in range(0, len(texts), BATCH_SIZE):
        batch = by_length[start:start + BATCH_SIZE]
        encoded = tokenizer([texts[i] for i in batch], truncation=True, max_length=MAX_LENGTH,
                            padding=True, return_tensors="pt").to(DEVICE)
        predictions[batch] = order[model(**encoded).logits.argmax(-1).cpu().numpy()]
    del model
    torch.cuda.empty_cache()
    return predictions


def predict_tfidf(repo, texts):
    bundle = joblib.load(hf_hub_download(repo, "tfidf-logreg.joblib", token=HF_TOKEN))
    return bundle["model"].predict(bundle["vectorizer"].transform(texts))


for name, repo in MODELS.items():
    predict = predict_tfidf if name == "TF-IDF" else predict_transformer
    lookup = dict(zip(unique_texts, predict(repo, unique_texts)))
    versions[name] = versions["text"].map(lookup).astype(int)
    print(f"{name:18} done")

versions.drop(columns="text").to_csv(RESULTS_DIR / "predictions.csv", index=False)   # ids only, no texts

## Results

- **AI texts caught:** share of AI texts (both classes) still called AI-generated or AI-assisted
- **Human texts flagged:** share of human texts called AI-generated or AI-assisted

In [ ]:
results = []
for name in MODELS:
    for version, part in versions.groupby("version"):
        is_ai = part["label"] > 0
        row = {"model": name, "version": version}
        if is_ai.any():
            row["AI texts caught"] = (part.loc[is_ai, name] > 0).mean()
            row["AI-generated caught"] = (part.loc[part["label"] == 1, name] > 0).mean()
            row["AI-assisted caught"] = (part.loc[part["label"] == 2, name] > 0).mean()
        if (~is_ai).any():
            row["Human texts flagged"] = (part.loc[~is_ai, name] > 0).mean()
        results.append(row)
results = pd.DataFrame(results)
results.round(4).to_csv(RESULTS_DIR / "attack-results.csv", index=False)

ai = results.pivot(index="model", columns="version", values="AI texts caught")[["original", "light", "strong"]]
human = results.pivot(index="model", columns="version", values="Human texts flagged")[["original", "formalized"]]
ai, human = ai.loc[list(MODELS)], human.loc[list(MODELS)]
print("AI texts caught:\n", ai.map("{:.1%}".format).to_string(), "\n")
print("Human texts flagged:\n", human.map("{:.1%}".format).to_string(), "\n")
print("AI-assisted texts caught:\n", results.pivot(index="model", columns="version", values="AI-assisted caught")
      [["original", "light", "strong"]].loc[list(MODELS)].map("{:.1%}".format).to_string())

## Charts

In [ ]:
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
SHADES = ["#9ec5f4", "#4a8fe0", "#1d5bab"]                    # one hue, light -> dark = original -> strong
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10.5})


def bars(table, title, filename, shades):
    fig, ax = plt.subplots(figsize=(9.2, 4.4), dpi=160)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    x, width = np.arange(len(table)), 0.8 / table.shape[1]
    for j, (column, color) in enumerate(zip(table.columns, shades)):
        values = table[column].to_numpy() * 100
        ax.bar(x + (j - (table.shape[1] - 1) / 2) * width, values, width * 0.92, color=color, label=column)
        for xi, v in zip(x, values):
            ax.text(xi + (j - (table.shape[1] - 1) / 2) * width, v + 1, f"{v:.0f}", ha="center", fontsize=8.5, color=INK)
    ax.set_xticks(x, table.index)
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
    for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("#b8b7b1"); ax.tick_params(colors=MUTED, length=0)
    ax.legend(frameon=False, ncol=table.shape[1], loc="upper left", bbox_to_anchor=(0, 1.12), labelcolor=INK)
    ax.set_title(title, loc="left", fontsize=12.5, color=INK, pad=28)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / filename, facecolor=SURFACE); plt.show()


bars(ai, "AI texts still caught after the humanizer attack", "ai-caught.png", SHADES)
bars(human, "Human texts wrongly flagged, before and after formalizing", "human-flagged.png", [SHADES[0], SHADES[2]])

## Download

In [ ]:
shutil.make_archive("results-humanizer-attack", "zip", root_dir=".", base_dir="results")
print("Download: results-humanizer-attack.zip")